In [ ]:
import os
from tfx.orchestration.experimental.interactive.interactive_context import InteractiveContext

# Inisialisasi konteks
interactive_context = InteractiveContext(pipeline_root='auliaanshari-pipeline')

# variabel global untuk file dan direktori
DATA_ROOT = 'data'
TRANSFORM_MODULE_FILE = 'resume_transform.py'
TUNER_MODULE_FILE = 'resume_tuner.py'
TRAINER_MODULE_FILE = 'resume_trainer.py'

In [2]:
%%writefile resume_transform.py
import tensorflow as tf
import tensorflow_transform as tft

LABEL_KEY = "category"
FEATURE_KEY = "resume_text"

def transformed_name(key):
    """Menambahkan suffix _xf pada fitur yang sudah ditransformasi"""
    return key + "_xf"

def preprocessing_fn(inputs):
    """
    Melakukan preprocessing pada raw features.
    """
    outputs = {}
    
    # 1. Mengubah teks resume menjadi lowercase
    outputs[transformed_name(FEATURE_KEY)] = tf.strings.lower(inputs[FEATURE_KEY])
    
    # 2. Mengubah label string (misal: "HR", "IT") menjadi index integer (0, 1, 2...)
    outputs[transformed_name(LABEL_KEY)] = tf.cast(inputs[LABEL_KEY], tf.int64)
    
    return outputs

Writing resume_transform.py


In [3]:

from tfx.components import CsvExampleGen, StatisticsGen, SchemaGen, ExampleValidator, Transform
from tfx.proto import example_gen_pb2

# ==========================================
# 1. DATA INGESTION (CsvExampleGen)
# ==========================================
output = example_gen_pb2.Output(
    split_config=example_gen_pb2.SplitConfig(splits=[
        example_gen_pb2.SplitConfig.Split(name="train", hash_buckets=8),
        example_gen_pb2.SplitConfig.Split(name="eval", hash_buckets=2)
    ])
)
example_gen = CsvExampleGen(input_base=DATA_ROOT, output_config=output)
interactive_context.run(example_gen)

ExecutionResult(
    component_id: CsvExampleGen
    execution_id: 1
    outputs:
        examples: OutputChannel(artifact_type=Examples, producer_component_id=CsvExampleGen, output_key=examples, additional_properties={}, additional_custom_properties={}))

In [8]:
from tfx.components import StatisticsGen, SchemaGen, ExampleValidator

# ==========================================
# 2. DATA VALIDATION
# ==========================================
statistics_gen = StatisticsGen(examples=example_gen.outputs['examples'])
interactive_context.run(statistics_gen)

schema_gen = SchemaGen(statistics=statistics_gen.outputs['statistics'])
interactive_context.run(schema_gen)

example_validator = ExampleValidator(
    statistics=statistics_gen.outputs['statistics'],
    schema=schema_gen.outputs['schema']
)
interactive_context.run(example_validator)


ExecutionResult(
    component_id: ExampleValidator
    execution_id: 13
    outputs:
        anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=ExampleValidator, output_key=anomalies, additional_properties={}, additional_custom_properties={}))

In [9]:
from tfx.components import Transform

# ==========================================
# 3. DATA PREPROCESSING (Transform)
# ==========================================
transform = Transform(
    examples=example_gen.outputs['examples'],
    schema=schema_gen.outputs['schema'],
    module_file=os.path.abspath(TRANSFORM_MODULE_FILE)
)
interactive_context.run(transform)

ExecutionResult(
    component_id: Transform
    execution_id: 14
    outputs:
        transform_graph: OutputChannel(artifact_type=TransformGraph, producer_component_id=Transform, output_key=transform_graph, additional_properties={}, additional_custom_properties={})
        transformed_examples: OutputChannel(artifact_type=Examples, producer_component_id=Transform, output_key=transformed_examples, additional_properties={}, additional_custom_properties={})
        updated_analyzer_cache: OutputChannel(artifact_type=TransformCache, producer_component_id=Transform, output_key=updated_analyzer_cache, additional_properties={}, additional_custom_properties={})
        pre_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=pre_transform_schema, additional_properties={}, additional_custom_properties={})
        pre_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=pre_transform_stats, additional_properties={}, additional_custom_properties={})
        post_transform_schema: OutputChannel(artifact_type=Schema, producer_component_id=Transform, output_key=post_transform_schema, additional_properties={}, additional_custom_properties={})
        post_transform_stats: OutputChannel(artifact_type=ExampleStatistics, producer_component_id=Transform, output_key=post_transform_stats, additional_properties={}, additional_custom_properties={})
        post_transform_anomalies: OutputChannel(artifact_type=ExampleAnomalies, producer_component_id=Transform, output_key=post_transform_anomalies, additional_properties={}, additional_custom_properties={}))

In [10]:
%%writefile resume_tuner.py
import keras_tuner as kt
import tensorflow as tf
import tensorflow_transform as tft
from typing import NamedTuple, Dict, Text, Any
from tfx.components.trainer.fn_args_utils import FnArgs

LABEL_KEY = "category"
FEATURE_KEY = "resume_text"
NUM_CLASSES = 24 

def transformed_name(key):
    return key + "_xf"

def gzip_reader_fn(filenames):
    return tf.data.TFRecordDataset(filenames, compression_type='GZIP')

def input_fn(file_pattern, tf_transform_output, num_epochs=None, batch_size=64):
    transform_feature_spec = (
        tf_transform_output.transformed_feature_spec().copy())
    
    dataset = tf.data.experimental.make_batched_features_dataset(
        file_pattern=file_pattern,
        batch_size=batch_size,
        features=transform_feature_spec,
        reader=gzip_reader_fn,
        num_epochs=num_epochs,
        label_key=transformed_name(LABEL_KEY))
    return dataset

TunerFnResult = NamedTuple('TunerFnResult', [('tuner', kt.Tuner), ('fit_kwargs', Dict[Text, Any])])

def tuner_fn(fn_args: FnArgs) -> TunerFnResult:
    tf_transform_output = tft.TFTransformOutput(fn_args.transform_graph_path)
    train_set = input_fn(fn_args.train_files, tf_transform_output, num_epochs=5)
    val_set = input_fn(fn_args.eval_files, tf_transform_output, num_epochs=5)

    VOCAB_SIZE = 4000 
    SEQUENCE_LENGTH = 300

    vectorize_layer = tf.keras.layers.TextVectorization(
        standardize="lower_and_strip_punctuation",
        max_tokens=VOCAB_SIZE,
        ngrams=2, 
        output_mode='int',
        output_sequence_length=SEQUENCE_LENGTH)
    
    train_texts = []
    for features, labels in train_set:
        text_batch = features[transformed_name(FEATURE_KEY)]
        for text_tensor in tf.reshape(text_batch, [-1]):
            raw_bytes = text_tensor.numpy()
            cleaned_text = raw_bytes.decode('utf-8', errors='ignore').encode('ascii', 'ignore').decode('ascii')
            train_texts.append(cleaned_text)
            
    vectorize_layer.adapt(train_texts)
    vocab = vectorize_layer.get_vocabulary()

    def model_builder(hp):
        embedding_dim = hp.Choice('embedding_dim', values=[64, 128])
        conv_filters = hp.Choice('conv_filters', values=[64, 128])
        lstm_units = hp.Choice('lstm_units', values=[32]) 
        dense_units = hp.Int('dense_units', min_value=64, max_value=128, step=64)
        learning_rate = hp.Choice('learning_rate', values=[1e-3, 5e-4])
        l2_rate = hp.Choice('l2_rate', values=[1e-4, 1e-5])

        model_vectorize_layer = tf.keras.layers.TextVectorization(
            vocabulary=vocab, 
            standardize="lower_and_strip_punctuation",
            max_tokens=VOCAB_SIZE,
            ngrams=2, 
            output_mode='int',
            output_sequence_length=SEQUENCE_LENGTH)

        inputs = tf.keras.Input(shape=(1,), name=transformed_name(FEATURE_KEY), dtype=tf.string)
        reshaped_narrative = tf.reshape(inputs, [-1])
        x = model_vectorize_layer(reshaped_narrative)
        
        x = tf.keras.layers.Embedding(VOCAB_SIZE, embedding_dim, name="embedding")(x)
        x = tf.keras.layers.SpatialDropout1D(0.3)(x)
        
        x = tf.keras.layers.Conv1D(filters=conv_filters, kernel_size=5, activation='relu')(x)
        x = tf.keras.layers.MaxPooling1D(pool_size=2)(x)
        
        x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(lstm_units, dropout=0.2, recurrent_dropout=0.2))(x)
        
        x = tf.keras.layers.Dense(dense_units, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(l2_rate))(x)
        x = tf.keras.layers.Dropout(0.5)(x)
        
        outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
        
        model = tf.keras.Model(inputs=inputs, outputs=outputs)
        model.compile(
            loss='sparse_categorical_crossentropy',
            optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
            metrics=['sparse_categorical_accuracy']
        )
        return model

    # Bayesian Optimization
    tuner = kt.BayesianOptimization(
        model_builder,
        objective='val_sparse_categorical_accuracy',
        max_trials=10, # Coba 10 trial cerdas
        num_initial_points=2, # 2 trial pertama acak, sisanya dianalisis secara matematis
        directory=fn_args.working_dir,
        project_name='resume_kt_bayesian'
    )

    fit_kwargs = {
        "x": train_set,
        "validation_data": val_set,
        "steps_per_epoch": fn_args.train_steps,
        "validation_steps": fn_args.eval_steps,
        "epochs": 5 
    }

    return TunerFnResult(tuner=tuner, fit_kwargs=fit_kwargs)

Writing resume_tuner.py


In [12]:
from tfx.components import Tuner
from tfx.proto import trainer_pb2

# ==========================================
# 4. TUNER
# ==========================================
tuner = Tuner(
    module_file=os.path.abspath(TUNER_MODULE_FILE),
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=20),
    eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=5)
)
interactive_context.run(tuner)

Trial 10 Complete [00h 00m 54s]
val_sparse_categorical_accuracy: 0.328125

Best val_sparse_categorical_accuracy So Far: 0.36250001192092896
Total elapsed time: 00h 08m 33s
Results summary
Results in aulia_anshari-pipeline\.temp\15\resume_kt_bayesian
Showing 10 best trials
Objective(name="val_sparse_categorical_accuracy", direction="max")

Trial 06 summary
Hyperparameters:
embedding_dim: 128
conv_filters: 128
lstm_units: 32
dense_units: 128
learning_rate: 0.001
l2_rate: 1e-05
Score: 0.36250001192092896

Trial 07 summary
Hyperparameters:
embedding_dim: 128
conv_filters: 128
lstm_units: 32
dense_units: 128
learning_rate: 0.001
l2_rate: 1e-05
Score: 0.3375000059604645

Trial 09 summary
Hyperparameters:
embedding_dim: 128
conv_filters: 128
lstm_units: 32
dense_units: 128
learning_rate: 0.001
l2_rate: 1e-05
Score: 0.328125

Trial 04 summary
Hyperparameters:
embedding_dim: 128
conv_filters: 128
lstm_units: 32
dense_units: 128
learning_rate: 0.001
l2_rate: 1e-05
Score: 0.265625

Trial 08 summa

ExecutionResult(
    component_id: Tuner
    execution_id: 15
    outputs:
        best_hyperparameters: OutputChannel(artifact_type=HyperParameters, producer_component_id=Tuner, output_key=best_hyperparameters, additional_properties={}, additional_custom_properties={})
        tuner_results: OutputChannel(artifact_type=TunerResults, producer_component_id=Tuner, output_key=tuner_results, additional_properties={}, additional_custom_properties={}))

In [13]:
%%writefile resume_trainer.py
import tensorflow as tf
import tensorflow_transform as tft
import os
from tfx.components.trainer.fn_args_utils import FnArgs

LABEL_KEY = "category"
FEATURE_KEY = "resume_text"
NUM_CLASSES = 24 

def transformed_name(key):
    return key + "_xf"

def gzip_reader_fn(filenames):
    return tf.data.TFRecordDataset(filenames, compression_type='GZIP')

def input_fn(file_pattern, tf_transform_output, num_epochs=None, batch_size=64):
    transform_feature_spec = (
        tf_transform_output.transformed_feature_spec().copy())
    
    dataset = tf.data.experimental.make_batched_features_dataset(
        file_pattern=file_pattern,
        batch_size=batch_size,
        features=transform_feature_spec,
        reader=gzip_reader_fn,
        num_epochs=num_epochs,
        label_key=transformed_name(LABEL_KEY))
    return dataset

def _get_serve_tf_examples_fn(model, tf_transform_output):
    model.tft_layer = tf_transform_output.transform_features_layer()
    
    @tf.function
    def serve_tf_examples_fn(serialized_tf_examples):
        feature_spec = tf_transform_output.raw_feature_spec()
        feature_spec.pop(LABEL_KEY)
        parsed_features = tf.io.parse_example(serialized_tf_examples, feature_spec)
        transformed_features = model.tft_layer(parsed_features)
        return model(transformed_features)
        
    return serve_tf_examples_fn

def run_fn(fn_args: FnArgs) -> None:
    tf_transform_output = tft.TFTransformOutput(fn_args.transform_graph_path)
    
    train_set = input_fn(fn_args.train_files, tf_transform_output, 20) 
    val_set = input_fn(fn_args.eval_files, tf_transform_output, 20)
    
    hparams = fn_args.hyperparameters.get('values')
    embedding_dim = hparams['embedding_dim']
    conv_filters = hparams['conv_filters']
    lstm_units = hparams['lstm_units']
    dense_units = hparams['dense_units']
    learning_rate = hparams['learning_rate']
    l2_rate = hparams.get('l2_rate', 1e-4)

    VOCAB_SIZE = 4000
    SEQUENCE_LENGTH = 300
    
    vectorize_layer = tf.keras.layers.TextVectorization(
        standardize="lower_and_strip_punctuation",
        max_tokens=VOCAB_SIZE,
        ngrams=2, 
        output_mode='int',
        output_sequence_length=SEQUENCE_LENGTH)
        
    train_texts = []
    for features, labels in train_set:
        text_batch = features[transformed_name(FEATURE_KEY)]
        for text_tensor in tf.reshape(text_batch, [-1]):
            raw_bytes = text_tensor.numpy()
            cleaned_text = raw_bytes.decode('utf-8', errors='ignore').encode('ascii', 'ignore').decode('ascii')
            train_texts.append(cleaned_text)
            
    vectorize_layer.adapt(train_texts)
    vocab = vectorize_layer.get_vocabulary()

    model_vectorize_layer = tf.keras.layers.TextVectorization(
        vocabulary=vocab,
        standardize="lower_and_strip_punctuation",
        max_tokens=VOCAB_SIZE,
        ngrams=2, 
        output_mode='int',
        output_sequence_length=SEQUENCE_LENGTH)

    inputs = tf.keras.Input(shape=(1,), name=transformed_name(FEATURE_KEY), dtype=tf.string)
    reshaped_narrative = tf.reshape(inputs, [-1])
    x = model_vectorize_layer(reshaped_narrative)
    
    x = tf.keras.layers.Embedding(VOCAB_SIZE, embedding_dim, name="embedding")(x)
    x = tf.keras.layers.SpatialDropout1D(0.3)(x)
    
    x = tf.keras.layers.Conv1D(filters=conv_filters, kernel_size=5, activation='relu')(x)
    x = tf.keras.layers.MaxPooling1D(pool_size=2)(x)
    
    x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(lstm_units, dropout=0.2, recurrent_dropout=0.2))(x)
    
    x = tf.keras.layers.Dense(dense_units, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(l2_rate))(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    
    lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
        initial_learning_rate=learning_rate,
        decay_steps=100,
        decay_rate=0.9)
    
    model.compile(
        loss='sparse_categorical_crossentropy',
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr_schedule),
        metrics=['sparse_categorical_accuracy']
    )
    
    log_dir = os.path.join(os.path.dirname(fn_args.serving_model_dir), 'logs')
    tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, update_freq='batch')
    es = tf.keras.callbacks.EarlyStopping(monitor='val_sparse_categorical_accuracy', mode='max', patience=5, restore_best_weights=True, verbose=1)
    
    model.fit(
        x=train_set,
        validation_data=val_set,
        callbacks=[tensorboard_callback, es],
        steps_per_epoch=fn_args.train_steps,
        validation_steps=fn_args.eval_steps,
        epochs=20
    )
    
    signatures = {
        'serving_default':
        _get_serve_tf_examples_fn(model, tf_transform_output).get_concrete_function(
            tf.TensorSpec(shape=[None], dtype=tf.string, name='examples')
        )
    }
    model.save(fn_args.serving_model_dir, save_format='tf', signatures=signatures)

Writing resume_trainer.py


In [14]:
from tfx.components import Trainer
from tfx.proto import trainer_pb2

# ==========================================
# 5. TRAINER
# ==========================================

trainer = Trainer(
    module_file=os.path.abspath(TRAINER_MODULE_FILE),
    examples=transform.outputs['transformed_examples'],
    transform_graph=transform.outputs['transform_graph'],
    schema=schema_gen.outputs['schema'],
    hyperparameters=tuner.outputs['best_hyperparameters'], # Menggunakan hasil dari Tuner
    train_args=trainer_pb2.TrainArgs(splits=['train'], num_steps=30),
    eval_args=trainer_pb2.EvalArgs(splits=['eval'], num_steps=5)
)
interactive_context.run(trainer)

Epoch 1/20
30/30 [==============================] - 19s 455ms/step - loss: 3.1670 - sparse_categorical_accuracy: 0.0510 - val_loss: 3.1500 - val_sparse_categorical_accuracy: 0.1000
Epoch 2/20
30/30 [==============================] - 13s 434ms/step - loss: 3.1179 - sparse_categorical_accuracy: 0.0688 - val_loss: 3.1015 - val_sparse_categorical_accuracy: 0.0844
Epoch 3/20
30/30 [==============================] - 13s 441ms/step - loss: 2.9276 - sparse_categorical_accuracy: 0.1792 - val_loss: 2.7604 - val_sparse_categorical_accuracy: 0.3187
Epoch 4/20
30/30 [==============================] - 14s 450ms/step - loss: 2.3498 - sparse_categorical_accuracy: 0.3609 - val_loss: 2.2285 - val_sparse_categorical_accuracy: 0.4187
Epoch 5/20
30/30 [==============================] - 13s 448ms/step - loss: 1.8296 - sparse_categorical_accuracy: 0.4802 - val_loss: 1.7792 - val_sparse_categorical_accuracy: 0.5375
Epoch 6/20
30/30 [==============================] - 14s 457ms/step - loss: 1.4471 - sparse_cate

INFO:tensorflow:struct2tensor is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_decision_forests is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:tensorflow_text is not available.


INFO:tensorflow:Assets written to: aulia_anshari-pipeline\Trainer\model\16\Format-Serving\assets


INFO:tensorflow:Assets written to: aulia_anshari-pipeline\Trainer\model\16\Format-Serving\assets


ExecutionResult(
    component_id: Trainer
    execution_id: 16
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=Trainer, output_key=model, additional_properties={}, additional_custom_properties={})
        model_run: OutputChannel(artifact_type=ModelRun, producer_component_id=Trainer, output_key=model_run, additional_properties={}, additional_custom_properties={}))

In [15]:
# Memuat ekstensi TensorBoard di Notebook
%load_ext tensorboard

# Arahkan logdir ke folder temp tempat Trainer menyimpan log-nya
%tensorboard --logdir aulia_anshari-pipeline/Trainer/model/

In [16]:
from tfx.dsl.components.common.resolver import Resolver
from tfx.dsl.input_resolution.strategies.latest_blessed_model_strategy import LatestBlessedModelStrategy
from tfx.types import Channel
from tfx.types.standard_artifacts import Model, ModelBlessing

# ==========================================
# 6. RESOLVER
# ==========================================
model_resolver = Resolver(
    strategy_class=LatestBlessedModelStrategy,
    model=Channel(type=Model),
    model_blessing=Channel(type=ModelBlessing)
).with_id('latest_blessed_model_resolver')

interactive_context.run(model_resolver)

ExecutionResult(
    component_id: latest_blessed_model_resolver
    execution_id: 17
    outputs:
        model: OutputChannel(artifact_type=Model, producer_component_id=latest_blessed_model_resolver, output_key=model, additional_properties={}, additional_custom_properties={})
        model_blessing: OutputChannel(artifact_type=ModelBlessing, producer_component_id=latest_blessed_model_resolver, output_key=model_blessing, additional_properties={}, additional_custom_properties={}))

In [17]:
import tensorflow_model_analysis as tfma
from tfx.components import Evaluator

# ==========================================
# 7. EVALUATOR (Memvalidasi Kualitas Model)
# ==========================================
# Konfigurasi Evaluasi
eval_config = tfma.EvalConfig(
    model_specs=[tfma.ModelSpec(label_key='category')],
    slicing_specs=[tfma.SlicingSpec()],
    metrics_specs=[
        tfma.MetricsSpec(metrics=[
            tfma.MetricConfig(class_name='ExampleCount'),
            # Kita gunakan SparseCategoricalAccuracy karena label kita angka 0-23
            tfma.MetricConfig(class_name='SparseCategoricalAccuracy',
                threshold=tfma.MetricThreshold(
                    value_threshold=tfma.GenericValueThreshold(
                        lower_bound={'value': 0.3}), # Syarat minimal akurasi 30%
                    change_threshold=tfma.GenericChangeThreshold(
                        direction=tfma.MetricDirection.HIGHER_IS_BETTER,
                        absolute={'value': 0.0001})
                )
            )
        ])
    ]
)

evaluator = Evaluator(
    examples=example_gen.outputs['examples'],
    model=trainer.outputs['model'],
    baseline_model=model_resolver.outputs['model'],
    eval_config=eval_config
)

interactive_context.run(evaluator)

interactive_context.show(evaluator.outputs['evaluation'])

Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


Instructions for updating:
Use eager execution and: 
`tf.data.TFRecordDataset(path)`


In [18]:
from tfx.components import Pusher
from tfx.proto import pusher_pb2

# ==========================================
# 8. PUSHER (Menyimpan Model ke Folder Deployment)
# ==========================================
pusher = Pusher(
    model=trainer.outputs['model'],
    model_blessing=evaluator.outputs['blessing'],
    push_destination=pusher_pb2.PushDestination(
        filesystem=pusher_pb2.PushDestination.Filesystem(
            base_directory='serving_model_dir/smart-ats-model'))
)

interactive_context.run(pusher)

ExecutionResult(
    component_id: Pusher
    execution_id: 19
    outputs:
        pushed_model: OutputChannel(artifact_type=PushedModel, producer_component_id=Pusher, output_key=pushed_model, additional_properties={}, additional_custom_properties={}))